# RAG Application Demo with Ollama and Qwen3.5:2b
## Using CV Document for Information Retrieval

Ollama Docker running in local with Qwen3.5:2b we need to use this as LLM provider for our RAG application. 

can run `docker run -d --gpus=all -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama:latest` 

and then check or pull the LLM from inside by running `docker exec -it ollama ollama list` if not there any model 

then run `docker exec -it ollama ollama pull qwen3.5:2b`

In [10]:
import os
from IPython.display import display
import ipywidgets as widgets
import json
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field, asdict
from datetime import datetime
import pandas as pd
import numpy as np

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# For streaming, use this instead:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

### Two ways you can provide the document for this RAG demo. 

One already kept one PDF in a folder and mention along with path. Or you can use the upload widget to upload a pdf to embed. 

In [12]:
# Step 1: Manually specify your PDF path (upload via Jupyter's file browser first)
pdf_path = "cv_dhiraj_patra.pdf"  # <-- Change this to your actual filename

if not os.path.exists(pdf_path):
    print(f"❌ File not found: {pdf_path}")
    print("👉 Upload your PDF using Jupyter's file browser (top-left folder icon)")
else:
    print(f"Processing {pdf_path}...")
    
    # Load and split
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    texts = text_splitter.split_documents(documents)
    print(f"Created {len(texts)} chunks.")
    
    # Embed and store
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db_path = "./chroma_db_cv"
    
    vectorstore = Chroma.from_documents(
        documents=texts, 
        embedding=embeddings, 
        persist_directory=db_path
    )
    # vectorstore.persist()
    
    print(f"✅ Success! Vector DB saved at: {os.path.abspath(db_path)}")

Processing cv_dhiraj_patra.pdf...
Created 19 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Success! Vector DB saved at: /home/dhiraj/Desktop/python/jupyter_notebooks/LLM/chroma_db_cv


In [13]:
def debug_context(inputs):
    """Prints retrieved docs and passes them through unchanged."""
    print("=" * 50)
    print("🔍 RETRIEVED CONTEXT:")
    if isinstance(inputs.get("context"), list):
        for i, doc in enumerate(inputs["context"]):
            print(f"[Chunk {i+1}] {doc.page_content[:300]}...")  # First 300 chars
    else:
        print(inputs["context"])
    print("=" * 50)
    return inputs  # Must return inputs to keep chain flowing


In [14]:
# # 1. Create Upload Widget and Process Button
# upload_btn = widgets.FileUpload(
#     accept='.pdf',
#     multiple=False,
#     description=' Upload CV'
# )

# process_btn = widgets.Button(
#     description='⚙️ Process & Vectorize',
#     button_style='success',
#     layout=widgets.Layout(width='200px')
# )

# output = widgets.Output()

# display(widgets.VBox([upload_btn, process_btn, output]))

In [15]:
# # 2. Processing Function
# def process_pdf(b):
#     with output:
#         output.clear_output()
        
#         if not upload_btn.value:
#             print(" No file uploaded! Please upload a PDF first.")
#             return
        
#         # Get uploaded file
#         file_name = list(upload_btn.value.keys())[0]
#         file_content = upload_btn.value[file_name]['content']
        
#         # Save temporarily
#         temp_path = f"temp_{file_name}"
#         with open(temp_path, 'wb') as f:
#             f.write(file_content)
            
#         print(f"📄 Processing {file_name}...")
        
#         try:
#             # Load PDF
#             loader = PyPDFLoader(temp_path)
#             documents = loader.load()
            
#             # Split text
#             text_splitter = RecursiveCharacterTextSplitter(
#                 chunk_size=500,
#                 chunk_overlap=50
#             )
#             texts = text_splitter.split_documents(documents)
#             print(f"✂️ Created {len(texts)} chunks.")
            
#             # Embeddings
#             print("🧠 Generating embeddings (this may take a moment)...")
#             embeddings = HuggingFaceEmbeddings(
#                 model_name="sentence-transformers/all-MiniLM-L6-v2"
#             )
            
#             # Vector Store
#             db_path = "./chroma_db_cv"
#             vectorstore = Chroma.from_documents(
#                 documents=texts, 
#                 embedding=embeddings, 
#                 persist_directory=db_path
#             )
#             vectorstore.persist()
            
#             print(f"✅ Success! Vector DB saved at: {os.path.abspath(db_path)}")
            
#         except Exception as e:
#             print(f"❌ Error: {str(e)}")
        
#         finally:
#             # Clean up temp file
#             if os.path.exists(temp_path):
#                 os.remove(temp_path)

# # 3. Attach to BUTTON click (not upload event)
# process_btn.on_click(process_pdf)

In [21]:
# 1. Load the existing Vector DB
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./chroma_db_cv", embedding_function=embeddings)

# 2. Create a Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

model = "qwen3.5:9b"

# 3. Initialize LLM
llm = ChatOllama(
    model=model,
    base_url="http://192.168.1.5:11434",
    temperature=0.2
)

# 4. Define Prompt
template = """You are an assistant for answering questions based on the provided context and as an expert on that context.
Context: {context}
Question: {question}
Answer:"""
prompt = PromptTemplate.from_template(template)

# 5. Create RAG Chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(debug_context)  # debug
    | prompt
    | llm
    | StrOutputParser()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
# 6. Ask a question
# Test retriever independently
query = "What are my top skills?"
docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

Retrieved 3 chunks:

--- Chunk 1 ---
Technical Roadmaps, Delivery Planning, Mentoring, Team Leadership
KEY SKILLS
LLMs, Generative AI, RAG, AI Agents, Vector databases, LangChain, LangGraph, Prompt Engineering, Fine-tuning 
(LoRA), Model evaluation, Guardrail, Embeddings, FAISS, Pinecone, Weaviate, MCP, Multi-Agent Systems, Agent 
Orchestration, Python, PyTorch, TensorFlow, Scikit-learn, MLOps, LLMOps, MLflow, CI/CD, AWS, Azure, GCP, SQL, 
Kubernetes, Docker, Databricks, Spark, Kafka, Snowflake, FastAPI, Flask, Django
Metadata: {'creator': 'Writer', 'page': 0, 'creationdate': '2026-08-27T12:31:49+05:30', 'producer': 'LibreOffice 24.2', 'source': 'cv_dhiraj_patra.pdf', 'author': 'python-docx', 'total_pages': 3, 'page_label': '1'}

--- Chunk 2 ---
Technical Roadmaps, Delivery Planning, Mentoring, Team Leadership
KEY SKILLS
LLMs, Generative AI, RAG, AI Agents, Vector databases, LangChain, LangGraph, Prompt Engineering, Fine-tuning 
(LoRA), Model evaluation, Guardrail, Embeddings, FAISS, P

In [23]:
query = "What are my top skills?"
response = rag_chain.invoke(query)
print(response)

🔍 RETRIEVED CONTEXT:
[Chunk 1] Technical Roadmaps, Delivery Planning, Mentoring, Team Leadership
KEY SKILLS
LLMs, Generative AI, RAG, AI Agents, Vector databases, LangChain, LangGraph, Prompt Engineering, Fine-tuning 
(LoRA), Model evaluation, Guardrail, Embeddings, FAISS, Pinecone, Weaviate, MCP, Multi-Agent Systems, Agent 
Orch...
[Chunk 2] Technical Roadmaps, Delivery Planning, Mentoring, Team Leadership
KEY SKILLS
LLMs, Generative AI, RAG, AI Agents, Vector databases, LangChain, LangGraph, Prompt Engineering, Fine-tuning 
(LoRA), Model evaluation, Guardrail, Embeddings, FAISS, Pinecone, Weaviate, MCP, Multi-Agent Systems, Agent 
Orch...
[Chunk 3] • Vicky Dhas - Digital Bank - Singapore
• Shourav Sharma - CEO Crypto and Gaming - Czechia
LANGUAGES
English – READ/WRITE/SPEAK - Professional Working Proficiency
German - READ/WRITE/SPEAK - Elementary Working Proficiency
French - READ/WRITE/SPEAK - Elementary Working Proficiency...


Based on the provided CV context, your top skills are listed under the **KEY SKILLS** section and include:

*   **Leadership & Planning:** Technical Roadmaps, Delivery Planning, Mentoring, Team Leadership
*   **AI & LLMs:** LLMs, Generative AI, RAG, AI Agents, Multi-Agent Systems, Agent Orchestration, MCP
*   **LLM Development & Tools:** LangChain, LangGraph, Prompt Engineering, Fine-tuning (LoRA), Model evaluation, Guardrail, Embeddings
*   **Vector Databases:** FAISS, Pinecone, Weaviate
*   **Programming & Frameworks:** Python, PyTorch, TensorFlow, Scikit-learn, FastAPI, Flask, Django
*   **MLOps & DevOps:** MLOps, LLMOps, MLflow, CI/CD, Kubernetes, Docker
*   **Cloud & Data Engineering:** AWS, Azure, GCP, SQL, Databricks, Spark, Kafka, Snowflake


In [24]:
query = "What kind of job role I can best suited for?"
response = rag_chain.invoke(query)
print(response)

🔍 RETRIEVED CONTEXT:
[Chunk 1] LEADERSHIP and PROFESSIONAL ENGAGEMENTS
Ownership-driven leader focused on product outcomes, mentoring teams, and driving innovation. 
Active participant in developer communities including Google Developer groups.
RECOMMENDATION FROM
• Nagendra Cherukupalli - IEEE Fellow and Renewable energy expert ...
[Chunk 2] LEADERSHIP and PROFESSIONAL ENGAGEMENTS
Ownership-driven leader focused on product outcomes, mentoring teams, and driving innovation. 
Active participant in developer communities including Google Developer groups.
RECOMMENDATION FROM
• Nagendra Cherukupalli - IEEE Fellow and Renewable energy expert ...
[Chunk 3] Technical Roadmaps, Delivery Planning, Mentoring, Team Leadership
KEY SKILLS
LLMs, Generative AI, RAG, AI Agents, Vector databases, LangChain, LangGraph, Prompt Engineering, Fine-tuning 
(LoRA), Model evaluation, Guardrail, Embeddings, FAISS, Pinecone, Weaviate, MCP, Multi-Agent Systems, Agent 
Orch...


Based on the provided CV context, here are the most suitable job roles for you:

## **Primary Job Roles:**

### 1. **AI/ML Engineering Lead**
- Leverage your expertise in LLMs, Generative AI, RAG, and AI Agents
- Apply MLOps, MLflow, CI/CD experience to productionize AI solutions

### 2. **Technical Team Leader / Engineering Manager**
- Your leadership profile (mentoring teams, driving innovation) aligns well with this role
- Experience in Technical Roadmaps and Delivery Planning is valuable here

### 3. **AI Solutions Architect**
- Strong foundation in Vector databases, LangChain, Agent Orchestration
- Multi-Agent Systems and Model evaluation skills are highly relevant

### 4. **LLM Operations Specialist (LLMOps)**
- Direct experience with LLMOps, Fine-tuning (LoRA), Guardrails
- Cloud platform expertise (AWS, Azure, GCP) supports this role

### 5. **AI Product Manager**
- Ownership-driven focus on product outcomes
- Technical roadmap planning and delivery experience

## **Key Strengt